In [1]:
import pandas as pd
import os
import subprocess

def parse_fasta_to_dataframe(fasta_file):
    # Initialize lists to hold the parsed data
    chromosomes = []
    starts = []
    ends = []
    sequences = []

    # Temporary variables to hold current sequence info
    current_sequence = ''
    current_header = ''

    with open(fasta_file, 'r') as file:
        for line in file:
            if line.startswith('>'):
                # Check and process the previous sequence if it's valid
                if current_sequence:
                    header_parts = current_header.split('_')
                    try:
                        start = int(header_parts[1])  # Try to convert the start to an integer
                        # If successful, append the data
                        chromosomes.append(header_parts[0][1:])  # Remove '>' and take the chromosome part
                        starts.append(start)
                        ends.append(start + 500)
                        sequences.append(current_sequence)
                    except ValueError:
                        # If conversion fails, skip this entry
                        pass
                    # Reset current_sequence for the next sequence
                    current_sequence = ''
                
                current_header = line.strip()
            else:
                current_sequence += line.strip()

        # Process the last entry if it's valid
        if current_sequence:
            header_parts = current_header.split('_')
            try:
                start = int(header_parts[1])  # Try to convert the start to an integer
                # If successful, append the data
                chromosomes.append(header_parts[0][1:])  # Remove '>' and take the chromosome part
                starts.append(start)
                ends.append(start + 499)
                sequences.append(current_sequence)
            except ValueError:
                # If conversion fails, this entry is skipped
                pass

    # Create DataFrame
    df_neg = pd.DataFrame({
        'chromosome': chromosomes,
        'start': starts,
        'end': ends,
        'sequence': sequences
    })

    return df_neg

def parse_fasta_to_dataframe_v2(fasta_file):
    # Initialize lists to hold the parsed data
    chromosomes = []
    starts = []
    ends = []
    sequences = []

    # Temporary variables to hold current sequence info
    current_sequence = ''
    current_header = ''

    with open(fasta_file, 'r') as file:
        for line in file:
            if line.startswith('>'):
                # Check and process the previous sequence if it's valid
                if current_sequence:
                    # Parse the header using the new format
                    header_parts = current_header[1:].split(':')  # Remove '>' and split by ':'
                    coord_parts = header_parts[1].split('-')  # Split the coordinates by '-'
                    
                    chromosome = header_parts[0]
                    start = int(coord_parts[0])
                    end = int(coord_parts[1])
                    
                    # Append the data
                    chromosomes.append(chromosome)
                    starts.append(start)
                    ends.append(end)
                    sequences.append(current_sequence)
                    
                    # Reset current_sequence for the next sequence
                    current_sequence = ''
                
                current_header = line.strip()
            else:
                current_sequence += line.strip()

        # Process the last entry if it's valid
        if current_sequence:
            # Parse the header using the new format
            header_parts = current_header[1:].split(':')  # Remove '>' and split by ':'
            coord_parts = header_parts[1].split('-')  # Split the coordinates by '-'
            
            chromosome = header_parts[0]
            start = int(coord_parts[0])
            end = int(coord_parts[1])
            
            # Append the data
            chromosomes.append(chromosome)
            starts.append(start)
            ends.append(end)
            sequences.append(current_sequence)

    # Create DataFrame
    df_neg = pd.DataFrame({
        'chromosome': chromosomes,
        'start': starts,
        'end': ends,
        'sequence': sequences
    })

    return df_neg

def dataframe_to_bed(df_neg, filename):
    """
    Export a DataFrame to a BED file.
    
    Parameters:
    - df_neg: pandas DataFrame with 'chromosome', 'start', and 'end' columns.
    - filename: String, the name of the file to save the BED data.
    """
    # Select only the necessary columns for a BED file
    bed_df = df_neg[['chromosome', 'start', 'end']]
    
    # Adjust start positions for BED format (0-based)
    bed_df['start'] = bed_df['start'] - 1
    
    # Export to BED without header and index
    bed_df.to_csv(filename, sep='\t', header=False, index=False)


def parse_bed_to_dataframe(bed_file):
    """
    Parse a BED file into a pandas DataFrame.
    
    Parameters:
    - bed_file: String, the path to the BED file.
    
    Returns:
    - df_neg: pandas DataFrame with 'chromosome', 'start', and 'end' columns.
    """
    # Load the BED file
    df_neg = pd.read_csv(bed_file, sep='\t', header=None, names=['chromosome', 'start', 'end'])
    
    # Adjust start positions from BED format (0-based) to 1-based
    df_neg['start'] = df_neg['start'] + 1
    
    return df_neg

def remove_matching_sequences(df_neg, df2):
    """
    Remove rows from df_neg where the sequence matches any sequence in df2.

    Parameters:
    - df_neg: pandas DataFrame, the first dataframe from which rows will be removed.
    - df2: pandas DataFrame, the second dataframe used as a reference for sequence matching.

    Returns:
    - df_neg: pandas DataFrame, the first dataframe with rows removed if their sequences match any in df2.
    """
    # Find sequences in df2 to create a set of unique sequences
    sequences_in_df2 = set(df2['sequence'])

    # Use DataFrame.isin() to filter rows in df_neg whose 'sequence' is not in df2
    df_neg_filtered = df_neg[~df_neg['sequence'].isin(sequences_in_df2)]

    return df_neg_filtered

def dataframe_to_fasta(df_neg, fasta_file):
    """
    Convert a pandas DataFrame to a FASTA file.
    
    Parameters:
    - df_neg: pandas DataFrame with 'chromosome', 'start', 'end', and 'sequence' columns.
    - fasta_file: String, the path to the output FASTA file.
    """
    with open(fasta_file, 'w') as f:
        for _, row in df_neg.iterrows():
            header = f">{row['chromosome']}:{row['start']}-{row['end']}"
            sequence = row['sequence']
            f.write(f"{header}\n{sequence}\n")

def process_fasta_files(fasta_file_neg,fasta_file_pos, peak_file_pos, output_fasta_file):
    """
    Processes input FASTA files to find non-overlapping sequences between
    negative and positive sets and exports the filtered sequences to a new FASTA file.
    
    Parameters:
    - fasta_file_neg: Path to the FASTA file containing negative sequences.
    - fasta_file_pos: Path to the FASTA file containing positive sequences.
    - output_fasta_file: Path for the output FASTA file with filtered sequences.
    """
    # Parse the negative FASTA file and remove duplicates
    df_neg = parse_fasta_to_dataframe(fasta_file_neg)
    df_neg = df_neg.drop_duplicates('sequence')
    
    # Export both DataFrames to BED files
    dataframe_to_bed(df_neg, 'df_neg.bed')
    
    # Find non-overlapping intervals using bedtools
    cmd = f"bedtools intersect -a df_neg.bed -b {peak_file_pos} -v > df_neg_no_overlap.bed"
    subprocess.run(cmd, shell=True, check=True)
    
    # Parse the no-overlap BED file and clean up
    df_neg_no_overlap = parse_bed_to_dataframe('df_neg_no_overlap.bed')

    # Filter the DataFrame based on the non-overlapping data
    df_filtered = df_neg.merge(df_neg_no_overlap, on=['chromosome', 'start', 'end'], how='inner')
    # Parse the positive FASTA file (assuming a different parsing function is needed)
    df_pos = parse_fasta_to_dataframe_v2(fasta_file_pos)
    df_filtered = remove_matching_sequences(df_filtered, df_pos)
    
    # Export the filtered DataFrame to a FASTA file
    dataframe_to_fasta(df_filtered, output_fasta_file)
    os.remove('df_neg.bed')
    os.remove('df_neg_no_overlap.bed')
    
    #
import pandas as pd

# Function to parse FASTA file
def parse_fasta(fasta_file):
    with open(fasta_file, 'r') as file:
        # Initialize variables
        sequences = []
        chr = []
        start = []
        end = []
        seq = ''
        
        # Read the FASTA file
        for line in file:
            if line.startswith('>'):
                # If not the first sequence, save the previous sequence
                if seq:
                    sequences.append(seq)
                    seq = ''
                # Parse header
                header = line[1:].strip().split(':')
                range_ = header[1].split('-')
                
                chr.append(header[0])
                start.append(range_[0])
                end.append(range_[1])
            else:
                # Accumulate sequence lines
                seq += line.strip()
        
        # Don't forget the last sequence
        if seq:
            sequences.append(seq)
    
    # Create DataFrame
    df = pd.DataFrame({
        'chr': chr,
        'start': start,
        'end': end,
        'sequence': sequences
    })
    
    return df

def df_to_fasta(df, output_file):
    with open(output_file, 'w') as f:
        for index, row in df.iterrows():
            # Format the header
            header = f">{row['chr']}:{row['start']}-{row['end']}\n"
            # Write the header and sequence to the file
            f.write(header)
            f.write(f"{row['sequence']}\n")

def separate_negatives(narrowpeak_list):
    for i, narrowpeak_file in enumerate(narrowpeak_list):
        df_tmp = pd.read_csv(narrowpeak_file, sep='\t', header=None)
        current_neg = order[i][['chr', 'start', 'end']]
        current_neg[6] = 0
        current_neg.columns = [0, 1, 2, 6]
        current_neg[[3, 4, 5, 7, 8, 9]] = "NA"
        current_neg = current_neg[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]]
        pd.concat([df_tmp, current_neg]).to_csv(narrowpeak_file[:-4] + ".withNeg.bed", sep='\t', header=None, index=None)
        current_neg.to_csv(narrowpeak_file[:-4]+".Neg.bed",index=None, header=None, sep = "\t")

        if narrowpeak_file.endswith("train.bed"):
            data_shuffled = current_neg.sample(frac=1).reset_index(drop=True)

            # Calculate the size of each split
            split_size = len(data_shuffled) // 4

            # Split the DataFrame into four parts
            part1 = data_shuffled.iloc[:split_size]
            part2 = data_shuffled.iloc[split_size:2*split_size]
            part3 = data_shuffled.iloc[2*split_size:3*split_size]
            part4 = data_shuffled.iloc[3*split_size:]

            # If you have remaining rows due to uneven division, you can distribute them:
            remaining_rows = len(data_shuffled) % 4
            if remaining_rows:
                part4 = pd.concat([part4, data_shuffled.iloc[-remaining_rows:]])
            parts = [part1,part2,part3,part4]

            for i in range(len(parts)):
                parts[i].to_csv(narrowpeak_file[:-4]+f".Neg_Part{str(i+1)}.bed",index=None, header=None, sep = "\t")



In [10]:
import os
import pandas as pd

# Example usage
fasta_file_neg = '../background_negatives_500bp_hg38/H1resting.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background.fasta'
fasta_file_pos = '../expand_peaks_500bp_fasta_hg38/H1resting.idr.optimal_peak.expand_500bp_hg38.fasta'
output_fasta_file = '../background_negatives_500bp_hg38/H1resting.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
peak_file_pos = '../peak_files/H1_resting/H1resting_all_possible_peaks_merged_expand500bp_eachside.bed'

process_fasta_files(fasta_file_neg, fasta_file_pos, peak_file_pos, output_fasta_file)

# Process and parse the FASTA file
fasta_file = '../background_negatives_500bp_hg38/H1resting.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
df = parse_fasta(fasta_file)

# Generate validation set
val = df[df['chr'] == 'chr4']
output_file_validation = '../background_negatives_500bp_hg38/H1resting.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta'
df_to_fasta(val, output_file_validation)

# Generate test set
test = df[(df['chr'] == 'chr8') | (df['chr'] == 'chr9')]
output_file_test = '../background_negatives_500bp_hg38/H1resting.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta'
df_to_fasta(test, output_file_test)

# Generate training set
train = df[~df['chr'].isin(['chr4', 'chr8', 'chr9'])]
output_file_train = '../background_negatives_500bp_hg38/H1resting.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
df_to_fasta(train, output_file_train)

# Prepare narrowPeak files
narrowpeak_list = [
    "../expand_peaks_500bp_hg38/H1resting.idr.optimal_peak.narrowPeak.train.bed",
    "../expand_peaks_500bp_hg38/H1resting.idr.optimal_peak.narrowPeak.validation.bed",
    "../expand_peaks_500bp_hg38/H1resting.idr.optimal_peak.narrowPeak.test.bed"
]
order = [train, val, test]

separate_negatives(narrowpeak_list)

# Prepare combined positive and negative FASTA files
neg_fasta_files = [
    '../background_negatives_500bp_hg38/H1resting.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta',
    '../background_negatives_500bp_hg38/H1resting.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta',
    '../background_negatives_500bp_hg38/H1resting.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
]
pos_fasta_files = [
    '../expand_peaks_500bp_fasta_hg38/H1resting.idr.optimal_peak.narrowPeak.validation.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/H1resting.idr.optimal_peak.narrowPeak.test.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/H1resting.idr.optimal_peak.narrowPeak.train.bed.fasta'
]
output_titles = [
    "../positive_negative_500bp_hg38/H1resting.idr.optimal_peak.narrowPeak.validation.posneg.fasta",
    "../positive_negative_500bp_hg38/H1resting.idr.optimal_peak.narrowPeak.test.posneg.fasta",
    "../positive_negative_500bp_hg38/H1resting.idr.optimal_peak.narrowPeak.train.posneg.fasta"
]

for i, (neg_fasta, pos_fasta, title) in enumerate(zip(neg_fasta_files, pos_fasta_files, output_titles)):
    os.system(f"cat {pos_fasta} {neg_fasta} > {title}")

/tmp/ipykernel_305000/174335090.py:138: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bed_df['start'] = bed_df['start'] - 1


In [9]:
import os
import pandas as pd

# Example usage
fasta_file_neg = '../background_negatives_500bp_hg38/H1stimulated.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background.fasta'
fasta_file_pos = '../expand_peaks_500bp_fasta_hg38/H1stimulated.idr.optimal_peak.expand_500bp_hg38.fasta'
output_fasta_file = '../background_negatives_500bp_hg38/H1stimulated.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
peak_file_pos = '../peak_files/H1_stimulated/H1stimulated_all_possible_peaks_merged_expand500bp_eachside.bed'

process_fasta_files(fasta_file_neg, fasta_file_pos, peak_file_pos, output_fasta_file)

# Process and parse the FASTA file
fasta_file = '../background_negatives_500bp_hg38/H1stimulated.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
df = parse_fasta(fasta_file)

# Generate validation set
val = df[df['chr'] == 'chr4']
output_file_validation = '../background_negatives_500bp_hg38/H1stimulated.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta'
df_to_fasta(val, output_file_validation)

# Generate test set
test = df[(df['chr'] == 'chr8') | (df['chr'] == 'chr9')]
output_file_test = '../background_negatives_500bp_hg38/H1stimulated.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta'
df_to_fasta(test, output_file_test)

# Generate training set
train = df[~df['chr'].isin(['chr4', 'chr8', 'chr9'])]
output_file_train = '../background_negatives_500bp_hg38/H1stimulated.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
df_to_fasta(train, output_file_train)

# Prepare narrowPeak files
narrowpeak_list = [
    "../expand_peaks_500bp_hg38/H1stimulated.idr.optimal_peak.narrowPeak.train.bed",
    "../expand_peaks_500bp_hg38/H1stimulated.idr.optimal_peak.narrowPeak.validation.bed",
    "../expand_peaks_500bp_hg38/H1stimulated.idr.optimal_peak.narrowPeak.test.bed"
]
order = [train, val, test]

separate_negatives(narrowpeak_list)

# Prepare combined positive and negative FASTA files
neg_fasta_files = [
    '../background_negatives_500bp_hg38/H1stimulated.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta',
    '../background_negatives_500bp_hg38/H1stimulated.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta',
    '../background_negatives_500bp_hg38/H1stimulated.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
]
pos_fasta_files = [
    '../expand_peaks_500bp_fasta_hg38/H1stimulated.idr.optimal_peak.narrowPeak.validation.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/H1stimulated.idr.optimal_peak.narrowPeak.test.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/H1stimulated.idr.optimal_peak.narrowPeak.train.bed.fasta'
]
output_titles = [
    "../positive_negative_500bp_hg38/H1stimulated.idr.optimal_peak.narrowPeak.validation.posneg.fasta",
    "../positive_negative_500bp_hg38/H1stimulated.idr.optimal_peak.narrowPeak.test.posneg.fasta",
    "../positive_negative_500bp_hg38/H1stimulated.idr.optimal_peak.narrowPeak.train.posneg.fasta"
]

for i, (neg_fasta, pos_fasta, title) in enumerate(zip(neg_fasta_files, pos_fasta_files, output_titles)):
    os.system(f"cat {pos_fasta} {neg_fasta} > {title}")

/tmp/ipykernel_305000/174335090.py:138: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bed_df['start'] = bed_df['start'] - 1


In [7]:
import os
import pandas as pd

# Example usage
fasta_file_neg = '../background_negatives_500bp_hg38/WTC11resting.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background.fasta'
fasta_file_pos = '../expand_peaks_500bp_fasta_hg38/WTC11resting.idr.optimal_peak.expand_500bp_hg38.fasta'
output_fasta_file = '../background_negatives_500bp_hg38/WTC11resting.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
peak_file_pos = '../peak_files/WTC11_resting/WTC11resting_all_possible_peaks_merged_expand500bp_eachside.bed'

process_fasta_files(fasta_file_neg, fasta_file_pos, peak_file_pos, output_fasta_file)

# Process and parse the FASTA file
fasta_file = '../background_negatives_500bp_hg38/WTC11resting.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
df = parse_fasta(fasta_file)

# Generate validation set
val = df[df['chr'] == 'chr4']
output_file_validation = '../background_negatives_500bp_hg38/WTC11resting.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta'
df_to_fasta(val, output_file_validation)

# Generate test set
test = df[(df['chr'] == 'chr8') | (df['chr'] == 'chr9')]
output_file_test = '../background_negatives_500bp_hg38/WTC11resting.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta'
df_to_fasta(test, output_file_test)

# Generate training set
train = df[~df['chr'].isin(['chr4', 'chr8', 'chr9'])]
output_file_train = '../background_negatives_500bp_hg38/WTC11resting.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
df_to_fasta(train, output_file_train)

# Prepare narrowPeak files
narrowpeak_list = [
    "../expand_peaks_500bp_hg38/WTC11resting.idr.optimal_peak.narrowPeak.train.bed",
    "../expand_peaks_500bp_hg38/WTC11resting.idr.optimal_peak.narrowPeak.validation.bed",
    "../expand_peaks_500bp_hg38/WTC11resting.idr.optimal_peak.narrowPeak.test.bed"
]
order = [train, val, test]

separate_negatives(narrowpeak_list)

# Prepare combined positive and negative FASTA files
neg_fasta_files = [
    '../background_negatives_500bp_hg38/WTC11resting.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta',
    '../background_negatives_500bp_hg38/WTC11resting.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta',
    '../background_negatives_500bp_hg38/WTC11resting.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
]
pos_fasta_files = [
    '../expand_peaks_500bp_fasta_hg38/WTC11resting.idr.optimal_peak.narrowPeak.validation.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/WTC11resting.idr.optimal_peak.narrowPeak.test.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/WTC11resting.idr.optimal_peak.narrowPeak.train.bed.fasta'
]
output_titles = [
    "../positive_negative_500bp_hg38/WTC11resting.idr.optimal_peak.narrowPeak.validation.posneg.fasta",
    "../positive_negative_500bp_hg38/WTC11resting.idr.optimal_peak.narrowPeak.test.posneg.fasta",
    "../positive_negative_500bp_hg38/WTC11resting.idr.optimal_peak.narrowPeak.train.posneg.fasta"
]

for i, (neg_fasta, pos_fasta, title) in enumerate(zip(neg_fasta_files, pos_fasta_files, output_titles)):
    os.system(f"cat {pos_fasta} {neg_fasta} > {title}")

/tmp/ipykernel_305000/174335090.py:138: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bed_df['start'] = bed_df['start'] - 1


In [8]:
import os
import pandas as pd

# Example usage
fasta_file_neg = '../background_negatives_500bp_hg38/WTC11stimulated.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background.fasta'
fasta_file_pos = '../expand_peaks_500bp_fasta_hg38/WTC11stimulated.idr.optimal_peak.expand_500bp_hg38.fasta'
output_fasta_file = '../background_negatives_500bp_hg38/WTC11stimulated.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
peak_file_pos = '../peak_files/WTC11_stimulated/WTC11stimulated_all_possible_peaks_merged_expand500bp_eachside.bed'

process_fasta_files(fasta_file_neg, fasta_file_pos, peak_file_pos, output_fasta_file)

# Process and parse the FASTA file
fasta_file = '../background_negatives_500bp_hg38/WTC11stimulated.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
df = parse_fasta(fasta_file)

# Generate validation set
val = df[df['chr'] == 'chr4']
output_file_validation = '../background_negatives_500bp_hg38/WTC11stimulated.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta'
df_to_fasta(val, output_file_validation)

# Generate test set
test = df[(df['chr'] == 'chr8') | (df['chr'] == 'chr9')]
output_file_test = '../background_negatives_500bp_hg38/WTC11stimulated.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta'
df_to_fasta(test, output_file_test)

# Generate training set
train = df[~df['chr'].isin(['chr4', 'chr8', 'chr9'])]
output_file_train = '../background_negatives_500bp_hg38/WTC11stimulated.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
df_to_fasta(train, output_file_train)

# Prepare narrowPeak files
narrowpeak_list = [
    "../expand_peaks_500bp_hg38/WTC11stimulated.idr.optimal_peak.narrowPeak.train.bed",
    "../expand_peaks_500bp_hg38/WTC11stimulated.idr.optimal_peak.narrowPeak.validation.bed",
    "../expand_peaks_500bp_hg38/WTC11stimulated.idr.optimal_peak.narrowPeak.test.bed"
]
order = [train, val, test]

separate_negatives(narrowpeak_list)

# Prepare combined positive and negative FASTA files
neg_fasta_files = [
    '../background_negatives_500bp_hg38/WTC11stimulated.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta',
    '../background_negatives_500bp_hg38/WTC11stimulated.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta',
    '../background_negatives_500bp_hg38/WTC11stimulated.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
]
pos_fasta_files = [
    '../expand_peaks_500bp_fasta_hg38/WTC11stimulated.idr.optimal_peak.narrowPeak.validation.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/WTC11stimulated.idr.optimal_peak.narrowPeak.test.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/WTC11stimulated.idr.optimal_peak.narrowPeak.train.bed.fasta'
]
output_titles = [
    "../positive_negative_500bp_hg38/WTC11stimulated.idr.optimal_peak.narrowPeak.validation.posneg.fasta",
    "../positive_negative_500bp_hg38/WTC11stimulated.idr.optimal_peak.narrowPeak.test.posneg.fasta",
    "../positive_negative_500bp_hg38/WTC11stimulated.idr.optimal_peak.narrowPeak.train.posneg.fasta"
]

for i, (neg_fasta, pos_fasta, title) in enumerate(zip(neg_fasta_files, pos_fasta_files, output_titles)):
    os.system(f"cat {pos_fasta} {neg_fasta} > {title}")

/tmp/ipykernel_305000/174335090.py:138: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bed_df['start'] = bed_df['start'] - 1


In [2]:
import os
import pandas as pd

# Example usage
fasta_file_neg = '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_IFNG_pos.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background.fasta'
fasta_file_pos = '../expand_peaks_500bp_fasta_hg38/THP1_LPSIFNG_vs_IFNG_pos.idr.optimal_peak.expand_500bp_hg38.fasta'
output_fasta_file = '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_IFNG_pos.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
peak_file_pos = '../peak_files/LPSIFNG_4hrs/LPSIFNGvsIFNG_all_possible_peaks_merged_expand500bp_eachside.bed'

process_fasta_files(fasta_file_neg, fasta_file_pos, peak_file_pos, output_fasta_file)

# Process and parse the FASTA file
fasta_file = '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_IFNG_pos.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
df = parse_fasta(fasta_file)

# Generate validation set
val = df[df['chr'] == 'chr4']
output_file_validation = '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_IFNG_pos.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta'
df_to_fasta(val, output_file_validation)

# Generate test set
test = df[(df['chr'] == 'chr8') | (df['chr'] == 'chr9')]
output_file_test = '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_IFNG_pos.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta'
df_to_fasta(test, output_file_test)

# Generate training set
train = df[~df['chr'].isin(['chr4', 'chr8', 'chr9'])]
output_file_train = '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_IFNG_pos.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
df_to_fasta(train, output_file_train)

# Prepare narrowPeak files
narrowpeak_list = [
    "../expand_peaks_500bp_hg38/THP1_LPSIFNG_vs_IFNG_pos.idr.optimal_peak.narrowPeak.train.bed",
    "../expand_peaks_500bp_hg38/THP1_LPSIFNG_vs_IFNG_pos.idr.optimal_peak.narrowPeak.validation.bed",
    "../expand_peaks_500bp_hg38/THP1_LPSIFNG_vs_IFNG_pos.idr.optimal_peak.narrowPeak.test.bed"
]
order = [train, val, test]

separate_negatives(narrowpeak_list)

# Prepare combined positive and negative FASTA files
neg_fasta_files = [
    '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_IFNG_pos.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta',
    '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_IFNG_pos.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta',
    '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_IFNG_pos.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
]
pos_fasta_files = [
    '../expand_peaks_500bp_fasta_hg38/THP1_LPSIFNG_vs_IFNG_pos.idr.optimal_peak.narrowPeak.validation.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/THP1_LPSIFNG_vs_IFNG_pos.idr.optimal_peak.narrowPeak.test.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/THP1_LPSIFNG_vs_IFNG_pos.idr.optimal_peak.narrowPeak.train.bed.fasta'
]
output_titles = [
    "../positive_negative_500bp_hg38/THP1_LPSIFNG_vs_IFNG_pos.idr.optimal_peak.narrowPeak.validation.posneg.fasta",
    "../positive_negative_500bp_hg38/THP1_LPSIFNG_vs_IFNG_pos.idr.optimal_peak.narrowPeak.test.posneg.fasta",
    "../positive_negative_500bp_hg38/THP1_LPSIFNG_vs_IFNG_pos.idr.optimal_peak.narrowPeak.train.posneg.fasta"
]

for i, (neg_fasta, pos_fasta, title) in enumerate(zip(neg_fasta_files, pos_fasta_files, output_titles)):
    os.system(f"cat {pos_fasta} {neg_fasta} > {title}")

/tmp/ipykernel_305000/174335090.py:138: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bed_df['start'] = bed_df['start'] - 1


In [6]:
import os
import pandas as pd

# Example usage
fasta_file_neg = '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_IFNG_neg.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background.fasta'
fasta_file_pos = '../expand_peaks_500bp_fasta_hg38/THP1_LPSIFNG_vs_IFNG_neg.idr.optimal_peak.expand_500bp_hg38.fasta'
output_fasta_file = '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_IFNG_neg.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
peak_file_pos = '../peak_files/LPSIFNG_4hrs/LPSIFNGvsIFNG_all_possible_peaks_merged_expand500bp_eachside.bed'

process_fasta_files(fasta_file_neg, fasta_file_pos, peak_file_pos, output_fasta_file)

# Process and parse the FASTA file
fasta_file = '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_IFNG_neg.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
df = parse_fasta(fasta_file)

# Generate validation set
val = df[df['chr'] == 'chr4']
output_file_validation = '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_IFNG_neg.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta'
df_to_fasta(val, output_file_validation)

# Generate test set
test = df[(df['chr'] == 'chr8') | (df['chr'] == 'chr9')]
output_file_test = '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_IFNG_neg.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta'
df_to_fasta(test, output_file_test)

# Generate training set
train = df[~df['chr'].isin(['chr4', 'chr8', 'chr9'])]
output_file_train = '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_IFNG_neg.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
df_to_fasta(train, output_file_train)

# Prepare narrowPeak files
narrowpeak_list = [
    "../expand_peaks_500bp_hg38/THP1_LPSIFNG_vs_IFNG_neg.idr.optimal_peak.narrowPeak.train.bed",
    "../expand_peaks_500bp_hg38/THP1_LPSIFNG_vs_IFNG_neg.idr.optimal_peak.narrowPeak.validation.bed",
    "../expand_peaks_500bp_hg38/THP1_LPSIFNG_vs_IFNG_neg.idr.optimal_peak.narrowPeak.test.bed"
]
order = [train, val, test]

separate_negatives(narrowpeak_list)

# Prepare combined positive and negative FASTA files
neg_fasta_files = [
    '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_IFNG_neg.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta',
    '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_IFNG_neg.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta',
    '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_IFNG_neg.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
]
pos_fasta_files = [
    '../expand_peaks_500bp_fasta_hg38/THP1_LPSIFNG_vs_IFNG_neg.idr.optimal_peak.narrowPeak.validation.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/THP1_LPSIFNG_vs_IFNG_neg.idr.optimal_peak.narrowPeak.test.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/THP1_LPSIFNG_vs_IFNG_neg.idr.optimal_peak.narrowPeak.train.bed.fasta'
]
output_titles = [
    "../positive_negative_500bp_hg38/THP1_LPSIFNG_vs_IFNG_neg.idr.optimal_peak.narrowPeak.validation.posneg.fasta",
    "../positive_negative_500bp_hg38/THP1_LPSIFNG_vs_IFNG_neg.idr.optimal_peak.narrowPeak.test.posneg.fasta",
    "../positive_negative_500bp_hg38/THP1_LPSIFNG_vs_IFNG_neg.idr.optimal_peak.narrowPeak.train.posneg.fasta"
]

for i, (neg_fasta, pos_fasta, title) in enumerate(zip(neg_fasta_files, pos_fasta_files, output_titles)):
    os.system(f"cat {pos_fasta} {neg_fasta} > {title}")

/tmp/ipykernel_305000/174335090.py:138: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bed_df['start'] = bed_df['start'] - 1


In [4]:
import os
import pandas as pd

# Example usage
fasta_file_neg = '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_Naive_pos.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background.fasta'
fasta_file_pos = '../expand_peaks_500bp_fasta_hg38/THP1_LPSIFNG_vs_Naive_pos.idr.optimal_peak.expand_500bp_hg38.fasta'
output_fasta_file = '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_Naive_pos.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
peak_file_pos = '../peak_files/LPSIFNG_4hrs/LPSIFNGvsNaive_all_possible_peaks_merged_expand500bp_eachside.bed'

process_fasta_files(fasta_file_neg, fasta_file_pos, peak_file_pos, output_fasta_file)

# Process and parse the FASTA file
fasta_file = '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_Naive_pos.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
df = parse_fasta(fasta_file)

# Generate validation set
val = df[df['chr'] == 'chr4']
output_file_validation = '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_Naive_pos.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta'
df_to_fasta(val, output_file_validation)

# Generate test set
test = df[(df['chr'] == 'chr8') | (df['chr'] == 'chr9')]
output_file_test = '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_Naive_pos.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta'
df_to_fasta(test, output_file_test)

# Generate training set
train = df[~df['chr'].isin(['chr4', 'chr8', 'chr9'])]
output_file_train = '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_Naive_pos.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
df_to_fasta(train, output_file_train)

# Prepare narrowPeak files
narrowpeak_list = [
    "../expand_peaks_500bp_hg38/THP1_LPSIFNG_vs_Naive_pos.idr.optimal_peak.narrowPeak.train.bed",
    "../expand_peaks_500bp_hg38/THP1_LPSIFNG_vs_Naive_pos.idr.optimal_peak.narrowPeak.validation.bed",
    "../expand_peaks_500bp_hg38/THP1_LPSIFNG_vs_Naive_pos.idr.optimal_peak.narrowPeak.test.bed"
]
order = [train, val, test]

separate_negatives(narrowpeak_list)

# Prepare combined positive and negative FASTA files
neg_fasta_files = [
    '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_Naive_pos.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta',
    '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_Naive_pos.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta',
    '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_Naive_pos.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
]
pos_fasta_files = [
    '../expand_peaks_500bp_fasta_hg38/THP1_LPSIFNG_vs_Naive_pos.idr.optimal_peak.narrowPeak.validation.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/THP1_LPSIFNG_vs_Naive_pos.idr.optimal_peak.narrowPeak.test.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/THP1_LPSIFNG_vs_Naive_pos.idr.optimal_peak.narrowPeak.train.bed.fasta'
]
output_titles = [
    "../positive_negative_500bp_hg38/THP1_LPSIFNG_vs_Naive_pos.idr.optimal_peak.narrowPeak.validation.posneg.fasta",
    "../positive_negative_500bp_hg38/THP1_LPSIFNG_vs_Naive_pos.idr.optimal_peak.narrowPeak.test.posneg.fasta",
    "../positive_negative_500bp_hg38/THP1_LPSIFNG_vs_Naive_pos.idr.optimal_peak.narrowPeak.train.posneg.fasta"
]

for i, (neg_fasta, pos_fasta, title) in enumerate(zip(neg_fasta_files, pos_fasta_files, output_titles)):
    os.system(f"cat {pos_fasta} {neg_fasta} > {title}")

/tmp/ipykernel_305000/174335090.py:138: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bed_df['start'] = bed_df['start'] - 1


In [5]:
import os
import pandas as pd

# Example usage
fasta_file_neg = '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_Naive_neg.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background.fasta'
fasta_file_pos = '../expand_peaks_500bp_fasta_hg38/THP1_LPSIFNG_vs_Naive_neg.idr.optimal_peak.expand_500bp_hg38.fasta'
output_fasta_file = '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_Naive_neg.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
peak_file_pos = '../peak_files/LPSIFNG_4hrs/LPSIFNGvsNaive_all_possible_peaks_merged_expand500bp_eachside.bed'

process_fasta_files(fasta_file_neg, fasta_file_pos, peak_file_pos, output_fasta_file)

# Process and parse the FASTA file
fasta_file = '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_Naive_neg.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
df = parse_fasta(fasta_file)

# Generate validation set
val = df[df['chr'] == 'chr4']
output_file_validation = '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_Naive_neg.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta'
df_to_fasta(val, output_file_validation)

# Generate test set
test = df[(df['chr'] == 'chr8') | (df['chr'] == 'chr9')]
output_file_test = '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_Naive_neg.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta'
df_to_fasta(test, output_file_test)

# Generate training set
train = df[~df['chr'].isin(['chr4', 'chr8', 'chr9'])]
output_file_train = '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_Naive_neg.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
df_to_fasta(train, output_file_train)

# Prepare narrowPeak files
narrowpeak_list = [
    "../expand_peaks_500bp_hg38/THP1_LPSIFNG_vs_Naive_neg.idr.optimal_peak.narrowPeak.train.bed",
    "../expand_peaks_500bp_hg38/THP1_LPSIFNG_vs_Naive_neg.idr.optimal_peak.narrowPeak.validation.bed",
    "../expand_peaks_500bp_hg38/THP1_LPSIFNG_vs_Naive_neg.idr.optimal_peak.narrowPeak.test.bed"
]
order = [train, val, test]

separate_negatives(narrowpeak_list)

# Prepare combined positive and negative FASTA files
neg_fasta_files = [
    '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_Naive_neg.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta',
    '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_Naive_neg.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta',
    '../background_negatives_500bp_hg38/THP1_LPSIFNG_vs_Naive_neg.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
]
pos_fasta_files = [
    '../expand_peaks_500bp_fasta_hg38/THP1_LPSIFNG_vs_Naive_neg.idr.optimal_peak.narrowPeak.validation.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/THP1_LPSIFNG_vs_Naive_neg.idr.optimal_peak.narrowPeak.test.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/THP1_LPSIFNG_vs_Naive_neg.idr.optimal_peak.narrowPeak.train.bed.fasta'
]
output_titles = [
    "../positive_negative_500bp_hg38/THP1_LPSIFNG_vs_Naive_neg.idr.optimal_peak.narrowPeak.validation.posneg.fasta",
    "../positive_negative_500bp_hg38/THP1_LPSIFNG_vs_Naive_neg.idr.optimal_peak.narrowPeak.test.posneg.fasta",
    "../positive_negative_500bp_hg38/THP1_LPSIFNG_vs_Naive_neg.idr.optimal_peak.narrowPeak.train.posneg.fasta"
]

for i, (neg_fasta, pos_fasta, title) in enumerate(zip(neg_fasta_files, pos_fasta_files, output_titles)):
    os.system(f"cat {pos_fasta} {neg_fasta} > {title}")

/tmp/ipykernel_305000/174335090.py:138: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bed_df['start'] = bed_df['start'] - 1


In [8]:
import os
import pandas as pd

# Example usage
fasta_file_neg = '../background_negatives_500bp_hg38/HEK293_ATAC_high_depth.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background.fasta'
fasta_file_pos = '../expand_peaks_500bp_fasta_hg38/HEK293_ATAC_high_depth.idr.optimal_peak.expand_500bp_hg38.fasta'
output_fasta_file = '../background_negatives_500bp_hg38/HEK293_ATAC_high_depth.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
peak_file_pos = '../peak_files/HEK293_ATAC/HEK293_ATAC_all_possible_peaks_merged_expand500bp_eachside.bed'

process_fasta_files(fasta_file_neg, fasta_file_pos, peak_file_pos, output_fasta_file)

# Process and parse the FASTA file
fasta_file = '../background_negatives_500bp_hg38/HEK293_ATAC_high_depth.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
df = parse_fasta(fasta_file)

# Generate validation set
val = df[df['chr'] == 'chr4']
output_file_validation = '../background_negatives_500bp_hg38/HEK293_ATAC_high_depth.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta'
df_to_fasta(val, output_file_validation)

# Generate test set
test = df[(df['chr'] == 'chr8') | (df['chr'] == 'chr9')]
output_file_test = '../background_negatives_500bp_hg38/HEK293_ATAC_high_depth.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta'
df_to_fasta(test, output_file_test)

# Generate training set
train = df[~df['chr'].isin(['chr4', 'chr8', 'chr9'])]
output_file_train = '../background_negatives_500bp_hg38/HEK293_ATAC_high_depth.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
df_to_fasta(train, output_file_train)

# Prepare narrowPeak files
narrowpeak_list = [
    "../expand_peaks_500bp_hg38/HEK293_ATAC_high_depth.idr.optimal_peak.narrowPeak.train.bed",
    "../expand_peaks_500bp_hg38/HEK293_ATAC_high_depth.idr.optimal_peak.narrowPeak.validation.bed",
    "../expand_peaks_500bp_hg38/HEK293_ATAC_high_depth.idr.optimal_peak.narrowPeak.test.bed"
]
order = [train, val, test]

separate_negatives(narrowpeak_list)

# Prepare combined positive and negative FASTA files
neg_fasta_files = [
    '../background_negatives_500bp_hg38/HEK293_ATAC_high_depth.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta',
    '../background_negatives_500bp_hg38/HEK293_ATAC_high_depth.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta',
    '../background_negatives_500bp_hg38/HEK293_ATAC_high_depth.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
]
pos_fasta_files = [
    '../expand_peaks_500bp_fasta_hg38/HEK293_ATAC_high_depth.idr.optimal_peak.narrowPeak.validation.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/HEK293_ATAC_high_depth.idr.optimal_peak.narrowPeak.test.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/HEK293_ATAC_high_depth.idr.optimal_peak.narrowPeak.train.bed.fasta'
]
output_titles = [
    "../positive_negative_500bp_hg38/HEK293_ATAC_high_depth.idr.optimal_peak.narrowPeak.validation.posneg.fasta",
    "../positive_negative_500bp_hg38/HEK293_ATAC_high_depth.idr.optimal_peak.narrowPeak.test.posneg.fasta",
    "../positive_negative_500bp_hg38/HEK293_ATAC_high_depth.idr.optimal_peak.narrowPeak.train.posneg.fasta"
]

for i, (neg_fasta, pos_fasta, title) in enumerate(zip(neg_fasta_files, pos_fasta_files, output_titles)):
    os.system(f"cat {pos_fasta} {neg_fasta} > {title}")

/tmp/ipykernel_27565/174335090.py:138: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bed_df['start'] = bed_df['start'] - 1


In [6]:
import os
import pandas as pd

# Example usage
fasta_file_neg = '../background_negatives_500bp_hg38/HEK293T_ATAC.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background.fasta'
fasta_file_pos = '../expand_peaks_500bp_fasta_hg38/HEK293T_ATAC.idr.optimal_peak.expand_500bp_hg38.fasta'
output_fasta_file = '../background_negatives_500bp_hg38/HEK293T_ATAC.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
peak_file_pos = '../peak_files/HEK293T_ATAC/HEK293T_ATAC_all_possible_peaks_merged_expand500bp_eachside.bed'

process_fasta_files(fasta_file_neg, fasta_file_pos, peak_file_pos, output_fasta_file)

# Process and parse the FASTA file
fasta_file = '../background_negatives_500bp_hg38/HEK293T_ATAC.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
df = parse_fasta(fasta_file)

# Generate validation set
val = df[df['chr'] == 'chr4']
output_file_validation = '../background_negatives_500bp_hg38/HEK293T_ATAC.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta'
df_to_fasta(val, output_file_validation)

# Generate test set
test = df[(df['chr'] == 'chr8') | (df['chr'] == 'chr9')]
output_file_test = '../background_negatives_500bp_hg38/HEK293T_ATAC.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta'
df_to_fasta(test, output_file_test)

# Generate training set
train = df[~df['chr'].isin(['chr4', 'chr8', 'chr9'])]
output_file_train = '../background_negatives_500bp_hg38/HEK293T_ATAC.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
df_to_fasta(train, output_file_train)

# Prepare narrowPeak files
narrowpeak_list = [
    "../expand_peaks_500bp_hg38/HEK293T_ATAC.idr.optimal_peak.narrowPeak.train.bed",
    "../expand_peaks_500bp_hg38/HEK293T_ATAC.idr.optimal_peak.narrowPeak.validation.bed",
    "../expand_peaks_500bp_hg38/HEK293T_ATAC.idr.optimal_peak.narrowPeak.test.bed"
]
order = [train, val, test]

separate_negatives(narrowpeak_list)

# Prepare combined positive and negative FASTA files
neg_fasta_files = [
    '../background_negatives_500bp_hg38/HEK293T_ATAC.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta',
    '../background_negatives_500bp_hg38/HEK293T_ATAC.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta',
    '../background_negatives_500bp_hg38/HEK293T_ATAC.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
]
pos_fasta_files = [
    '../expand_peaks_500bp_fasta_hg38/HEK293T_ATAC.idr.optimal_peak.narrowPeak.validation.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/HEK293T_ATAC.idr.optimal_peak.narrowPeak.test.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/HEK293T_ATAC.idr.optimal_peak.narrowPeak.train.bed.fasta'
]
output_titles = [
    "../positive_negative_500bp_hg38/HEK293T_ATAC.idr.optimal_peak.narrowPeak.validation.posneg.fasta",
    "../positive_negative_500bp_hg38/HEK293T_ATAC.idr.optimal_peak.narrowPeak.test.posneg.fasta",
    "../positive_negative_500bp_hg38/HEK293T_ATAC.idr.optimal_peak.narrowPeak.train.posneg.fasta"
]

for i, (neg_fasta, pos_fasta, title) in enumerate(zip(neg_fasta_files, pos_fasta_files, output_titles)):
    os.system(f"cat {pos_fasta} {neg_fasta} > {title}")

/tmp/ipykernel_27565/174335090.py:138: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bed_df['start'] = bed_df['start'] - 1


In [ ]:
import os
import pandas as pd

# Example usage
fasta_file_neg = '../background_negatives_500bp_hg38/THP1_Naive_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background.fasta'
fasta_file_pos = '../expand_peaks_500bp_fasta_hg38/THP1_Naive_4hrs.idr.optimal_peak.expand_500bp_hg38.fasta'
output_fasta_file = '../background_negatives_500bp_hg38/THP1_Naive_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
peak_file_pos = '../peak_files/Naive_4hrs/Naive_all_possible_peaks_merged_expand500bp_eachside.bed'

process_fasta_files(fasta_file_neg, fasta_file_pos, peak_file_pos, output_fasta_file)

# Process and parse the FASTA file
fasta_file = '../background_negatives_500bp_hg38/THP1_Naive_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
df = parse_fasta(fasta_file)

# Generate validation set
val = df[df['chr'] == 'chr4']
output_file_validation = '../background_negatives_500bp_hg38/THP1_Naive_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta'
df_to_fasta(val, output_file_validation)

# Generate test set
test = df[(df['chr'] == 'chr8') | (df['chr'] == 'chr9')]
output_file_test = '../background_negatives_500bp_hg38/THP1_Naive_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta'
df_to_fasta(test, output_file_test)

# Generate training set
train = df[~df['chr'].isin(['chr4', 'chr8', 'chr9'])]
output_file_train = '../background_negatives_500bp_hg38/THP1_Naive_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
df_to_fasta(train, output_file_train)

# Prepare narrowPeak files
narrowpeak_list = [
    "../expand_peaks_500bp_hg38/THP1_Naive_4hrs.idr.optimal_peak.narrowPeak.train.bed",
    "../expand_peaks_500bp_hg38/THP1_Naive_4hrs.idr.optimal_peak.narrowPeak.validation.bed",
    "../expand_peaks_500bp_hg38/THP1_Naive_4hrs.idr.optimal_peak.narrowPeak.test.bed"
]
order = [train, val, test]

separate_negatives(narrowpeak_list)

# Prepare combined positive and negative FASTA files
neg_fasta_files = [
    '../background_negatives_500bp_hg38/THP1_Naive_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta',
    '../background_negatives_500bp_hg38/THP1_Naive_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta',
    '../background_negatives_500bp_hg38/THP1_Naive_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
]
pos_fasta_files = [
    '../expand_peaks_500bp_fasta_hg38/THP1_Naive_4hrs.idr.optimal_peak.narrowPeak.validation.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/THP1_Naive_4hrs.idr.optimal_peak.narrowPeak.test.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/THP1_Naive_4hrs.idr.optimal_peak.narrowPeak.train.bed.fasta'
]
output_titles = [
    "../positive_negative_500bp_hg38/THP1_Naive_4hrs.idr.optimal_peak.narrowPeak.validation.posneg.fasta",
    "../positive_negative_500bp_hg38/THP1_Naive_4hrs.idr.optimal_peak.narrowPeak.test.posneg.fasta",
    "../positive_negative_500bp_hg38/THP1_Naive_4hrs.idr.optimal_peak.narrowPeak.train.posneg.fasta"
]

for i, (neg_fasta, pos_fasta, title) in enumerate(zip(neg_fasta_files, pos_fasta_files, output_titles)):
    os.system(f"cat {pos_fasta} {neg_fasta} > {title}")

In [3]:
import os
import pandas as pd

# Example usage
fasta_file_neg = '../background_negatives_500bp_hg38/THP1_IFNG_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background.fasta'
fasta_file_pos = '../expand_peaks_500bp_fasta_hg38/THP1_IFNG_4hrs.idr.optimal_peak.expand_500bp_hg38.fasta'
output_fasta_file = '../background_negatives_500bp_hg38/THP1_IFNG_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
peak_file_pos = '../peak_files/Naive_4hrs/Naive_all_possible_peaks_merged_expand500bp_eachside.bed'

process_fasta_files(fasta_file_neg, fasta_file_pos, peak_file_pos, output_fasta_file)

# Process and parse the FASTA file
fasta_file = '../background_negatives_500bp_hg38/THP1_IFNG_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
df = parse_fasta(fasta_file)

# Generate validation set
val = df[df['chr'] == 'chr4']
output_file_validation = '../background_negatives_500bp_hg38/THP1_IFNG_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta'
df_to_fasta(val, output_file_validation)

# Generate test set
test = df[(df['chr'] == 'chr8') | (df['chr'] == 'chr9')]
output_file_test = '../background_negatives_500bp_hg38/THP1_IFNG_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta'
df_to_fasta(test, output_file_test)

# Generate training set
train = df[~df['chr'].isin(['chr4', 'chr8', 'chr9'])]
output_file_train = '../background_negatives_500bp_hg38/THP1_IFNG_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
df_to_fasta(train, output_file_train)

# Prepare narrowPeak files
narrowpeak_list = [
    "../expand_peaks_500bp_hg38/THP1_IFNG_4hrs.idr.optimal_peak.narrowPeak.train.bed",
    "../expand_peaks_500bp_hg38/THP1_IFNG_4hrs.idr.optimal_peak.narrowPeak.validation.bed",
    "../expand_peaks_500bp_hg38/THP1_IFNG_4hrs.idr.optimal_peak.narrowPeak.test.bed"
]
order = [train, val, test]

separate_negatives(narrowpeak_list)

# Prepare combined positive and negative FASTA files
neg_fasta_files = [
    '../background_negatives_500bp_hg38/THP1_IFNG_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta',
    '../background_negatives_500bp_hg38/THP1_IFNG_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta',
    '../background_negatives_500bp_hg38/THP1_IFNG_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
]
pos_fasta_files = [
    '../expand_peaks_500bp_fasta_hg38/THP1_IFNG_4hrs.idr.optimal_peak.narrowPeak.validation.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/THP1_IFNG_4hrs.idr.optimal_peak.narrowPeak.test.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/THP1_IFNG_4hrs.idr.optimal_peak.narrowPeak.train.bed.fasta'
]
output_titles = [
    "../positive_negative_500bp_hg38/THP1_IFNG_4hrs.idr.optimal_peak.narrowPeak.validation.posneg.fasta",
    "../positive_negative_500bp_hg38/THP1_IFNG_4hrs.idr.optimal_peak.narrowPeak.test.posneg.fasta",
    "../positive_negative_500bp_hg38/THP1_IFNG_4hrs.idr.optimal_peak.narrowPeak.train.posneg.fasta"
]

for i, (neg_fasta, pos_fasta, title) in enumerate(zip(neg_fasta_files, pos_fasta_files, output_titles)):
    os.system(f"cat {pos_fasta} {neg_fasta} > {title}")

/tmp/ipykernel_4156/174335090.py:138: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bed_df['start'] = bed_df['start'] - 1


In [5]:
import os
import pandas as pd

# Example usage
fasta_file_neg = '../background_negatives_500bp_hg38/THP1_IFNB_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background.fasta'
fasta_file_pos = '../expand_peaks_500bp_fasta_hg38/THP1_IFNB_4hrs.idr.optimal_peak.expand_500bp_hg38.fasta'
output_fasta_file = '../background_negatives_500bp_hg38/THP1_IFNB_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
peak_file_pos = '../peak_files/Naive_4hrs/Naive_all_possible_peaks_merged_expand500bp_eachside.bed'

process_fasta_files(fasta_file_neg, fasta_file_pos, peak_file_pos, output_fasta_file)

# Process and parse the FASTA file
fasta_file = '../background_negatives_500bp_hg38/THP1_IFNB_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
df = parse_fasta(fasta_file)

# Generate validation set
val = df[df['chr'] == 'chr4']
output_file_validation = '../background_negatives_500bp_hg38/THP1_IFNB_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta'
df_to_fasta(val, output_file_validation)

# Generate test set
test = df[(df['chr'] == 'chr8') | (df['chr'] == 'chr9')]
output_file_test = '../background_negatives_500bp_hg38/THP1_IFNB_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta'
df_to_fasta(test, output_file_test)

# Generate training set
train = df[~df['chr'].isin(['chr4', 'chr8', 'chr9'])]
output_file_train = '../background_negatives_500bp_hg38/THP1_IFNB_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
df_to_fasta(train, output_file_train)

# Prepare narrowPeak files
narrowpeak_list = [
    "../expand_peaks_500bp_hg38/THP1_IFNB_4hrs.idr.optimal_peak.narrowPeak.train.bed",
    "../expand_peaks_500bp_hg38/THP1_IFNB_4hrs.idr.optimal_peak.narrowPeak.validation.bed",
    "../expand_peaks_500bp_hg38/THP1_IFNB_4hrs.idr.optimal_peak.narrowPeak.test.bed"
]
order = [train, val, test]

separate_negatives(narrowpeak_list)

# Prepare combined positive and negative FASTA files
neg_fasta_files = [
    '../background_negatives_500bp_hg38/THP1_IFNB_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta',
    '../background_negatives_500bp_hg38/THP1_IFNB_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta',
    '../background_negatives_500bp_hg38/THP1_IFNB_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
]
pos_fasta_files = [
    '../expand_peaks_500bp_fasta_hg38/THP1_IFNB_4hrs.idr.optimal_peak.narrowPeak.validation.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/THP1_IFNB_4hrs.idr.optimal_peak.narrowPeak.test.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/THP1_IFNB_4hrs.idr.optimal_peak.narrowPeak.train.bed.fasta'
]
output_titles = [
    "../positive_negative_500bp_hg38/THP1_IFNB_4hrs.idr.optimal_peak.narrowPeak.validation.posneg.fasta",
    "../positive_negative_500bp_hg38/THP1_IFNB_4hrs.idr.optimal_peak.narrowPeak.test.posneg.fasta",
    "../positive_negative_500bp_hg38/THP1_IFNB_4hrs.idr.optimal_peak.narrowPeak.train.posneg.fasta"
]

for i, (neg_fasta, pos_fasta, title) in enumerate(zip(neg_fasta_files, pos_fasta_files, output_titles)):
    os.system(f"cat {pos_fasta} {neg_fasta} > {title}")

/tmp/ipykernel_4156/174335090.py:138: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  bed_df['start'] = bed_df['start'] - 1


In [ ]:
import os
import pandas as pd

# Example usage
fasta_file_neg = '../background_negatives_500bp_hg38/THP1_LPSIFNG_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background.fasta'
fasta_file_pos = '../expand_peaks_500bp_fasta_hg38/THP1_LPSIFNG_4hrs.idr.optimal_peak.expand_500bp_hg38.fasta'
output_fasta_file = '../background_negatives_500bp_hg38/THP1_LPSIFNG_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
peak_file_pos = '../peak_files/LPSIFNG_4hrs/LPSIFNG_all_possible_peaks_merged_expand500bp_eachside.bed'

process_fasta_files(fasta_file_neg, fasta_file_pos, peak_file_pos, output_fasta_file)

# Process and parse the FASTA file
fasta_file = '../background_negatives_500bp_hg38/THP1_LPSIFNG_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
df = parse_fasta(fasta_file)

# Generate validation set
val = df[df['chr'] == 'chr4']
output_file_validation = '../background_negatives_500bp_hg38/THP1_LPSIFNG_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta'
df_to_fasta(val, output_file_validation)

# Generate test set
test = df[(df['chr'] == 'chr8') | (df['chr'] == 'chr9')]
output_file_test = '../background_negatives_500bp_hg38/THP1_LPSIFNG_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta'
df_to_fasta(test, output_file_test)

# Generate training set
train = df[~df['chr'].isin(['chr4', 'chr8', 'chr9'])]
output_file_train = '../background_negatives_500bp_hg38/THP1_LPSIFNG_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
df_to_fasta(train, output_file_train)

# Prepare narrowPeak files
narrowpeak_list = [
    "../expand_peaks_500bp_hg38/THP1_LPSIFNG_4hrs.idr.optimal_peak.narrowPeak.train.bed",
    "../expand_peaks_500bp_hg38/THP1_LPSIFNG_4hrs.idr.optimal_peak.narrowPeak.validation.bed",
    "../expand_peaks_500bp_hg38/THP1_LPSIFNG_4hrs.idr.optimal_peak.narrowPeak.test.bed"
]
order = [train, val, test]

separate_negatives(narrowpeak_list)

# Prepare combined positive and negative FASTA files
neg_fasta_files = [
    '../background_negatives_500bp_hg38/THP1_LPSIFNG_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta',
    '../background_negatives_500bp_hg38/THP1_LPSIFNG_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta',
    '../background_negatives_500bp_hg38/THP1_LPSIFNG_4hrs.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
]
pos_fasta_files = [
    '../expand_peaks_500bp_fasta_hg38/THP1_LPSIFNG_4hrs.idr.optimal_peak.narrowPeak.validation.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/THP1_LPSIFNG_4hrs.idr.optimal_peak.narrowPeak.test.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/THP1_LPSIFNG_4hrs.idr.optimal_peak.narrowPeak.train.bed.fasta'
]
output_titles = [
    "../positive_negative_500bp_hg38/THP1_LPSIFNG_4hrs.idr.optimal_peak.narrowPeak.validation.posneg.fasta",
    "../positive_negative_500bp_hg38/THP1_LPSIFNG_4hrs.idr.optimal_peak.narrowPeak.test.posneg.fasta",
    "../positive_negative_500bp_hg38/THP1_LPSIFNG_4hrs.idr.optimal_peak.narrowPeak.train.posneg.fasta"
]

for i, (neg_fasta, pos_fasta, title) in enumerate(zip(neg_fasta_files, pos_fasta_files, output_titles)):
    os.system(f"cat {pos_fasta} {neg_fasta} > {title}")

In [ ]:
import os
import pandas as pd

# Example usage
fasta_file_neg = '../background_negatives_500bp_hg38/THP1_monocytes.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background.fasta'
fasta_file_pos = '../expand_peaks_500bp_fasta_hg38/THP1_monocytes.idr.optimal_peak.expand_500bp_hg38.fasta'
output_fasta_file = '../background_negatives_500bp_hg38/THP1_monocytes.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
peak_file_pos = '../peak_files/THP1_Monocytes/THP1Monocyte_all_possible_peaks_merged_expand500bp_eachside.bed'

process_fasta_files(fasta_file_neg, fasta_file_pos, peak_file_pos, output_fasta_file)

# Process and parse the FASTA file
fasta_file = '../background_negatives_500bp_hg38/THP1_monocytes.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
df = parse_fasta(fasta_file)

# Generate validation set
val = df[df['chr'] == 'chr4']
output_file_validation = '../background_negatives_500bp_hg38/THP1_monocytes.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta'
df_to_fasta(val, output_file_validation)

# Generate test set
test = df[(df['chr'] == 'chr8') | (df['chr'] == 'chr9')]
output_file_test = '../background_negatives_500bp_hg38/THP1_monocytes.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta'
df_to_fasta(test, output_file_test)

# Generate training set
train = df[~df['chr'].isin(['chr4', 'chr8', 'chr9'])]
output_file_train = '../background_negatives_500bp_hg38/THP1_monocytes.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
df_to_fasta(train, output_file_train)

# Prepare narrowPeak files
narrowpeak_list = [
    "../expand_peaks_500bp_hg38/THP1_monocytes.idr.optimal_peak.narrowPeak.train.bed",
    "../expand_peaks_500bp_hg38/THP1_monocytes.idr.optimal_peak.narrowPeak.validation.bed",
    "../expand_peaks_500bp_hg38/THP1_monocytes.idr.optimal_peak.narrowPeak.test.bed"
]
order = [train, val, test]

separate_negatives(narrowpeak_list)

# Prepare combined positive and negative FASTA files
neg_fasta_files = [
    '../background_negatives_500bp_hg38/THP1_monocytes.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta',
    '../background_negatives_500bp_hg38/THP1_monocytes.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta',
    '../background_negatives_500bp_hg38/THP1_monocytes.idr.optimal_peak.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
]
pos_fasta_files = [
    '../expand_peaks_500bp_fasta_hg38/THP1_monocytes.idr.optimal_peak.narrowPeak.validation.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/THP1_monocytes.idr.optimal_peak.narrowPeak.test.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/THP1_monocytes.idr.optimal_peak.narrowPeak.train.bed.fasta'
]
output_titles = [
    "../positive_negative_500bp_hg38/THP1_monocytes.idr.optimal_peak.narrowPeak.validation.posneg.fasta",
    "../positive_negative_500bp_hg38/THP1_monocytes.idr.optimal_peak.narrowPeak.test.posneg.fasta",
    "../positive_negative_500bp_hg38/THP1_monocytes.idr.optimal_peak.narrowPeak.train.posneg.fasta"
]

for i, (neg_fasta, pos_fasta, title) in enumerate(zip(neg_fasta_files, pos_fasta_files, output_titles)):
    os.system(f"cat {pos_fasta} {neg_fasta} > {title}")

In [ ]:
import os
import pandas as pd

# Example usage
fasta_file_neg = '../background_negatives_500bp_hg38/HEK293T_DNase_ENCODE.expand_500bp_hg38_biasaway_GCmatch_background.fasta'
fasta_file_pos = '../expand_peaks_500bp_fasta_hg38/HEK293T_DNase_ENCODE.expand_500bp_hg38.fasta'
output_fasta_file = '../background_negatives_500bp_hg38/HEK293T_DNase_ENCODE.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
peak_file_pos = '../peak_files/HEK293T/HEK293T_all_possible_peaks_merged_expand500bp_eachside.bed'

process_fasta_files(fasta_file_neg, fasta_file_pos, peak_file_pos, output_fasta_file)

# Process and parse the FASTA file
fasta_file = '../background_negatives_500bp_hg38/HEK293T_DNase_ENCODE.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap.fasta'
df = parse_fasta(fasta_file)

# Generate validation set
val = df[df['chr'] == 'chr4']
output_file_validation = '../background_negatives_500bp_hg38/HEK293T_DNase_ENCODE.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta'
df_to_fasta(val, output_file_validation)

# Generate test set
test = df[(df['chr'] == 'chr8') | (df['chr'] == 'chr9')]
output_file_test = '../background_negatives_500bp_hg38/HEK293T_DNase_ENCODE.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta'
df_to_fasta(test, output_file_test)

# Generate training set
train = df[~df['chr'].isin(['chr4', 'chr8', 'chr9'])]
output_file_train = '../background_negatives_500bp_hg38/HEK293T_DNase_ENCODE.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
df_to_fasta(train, output_file_train)

# Prepare narrowPeak files
narrowpeak_list = [
    "../expand_peaks_500bp_hg38/HEK293T_DNase_ENCODE.narrowPeak.train.bed",
    "../expand_peaks_500bp_hg38/HEK293T_DNase_ENCODE.narrowPeak.validation.bed",
    "../expand_peaks_500bp_hg38/HEK293T_DNase_ENCODE.narrowPeak.test.bed"
]
order = [train, val, test]

separate_negatives(narrowpeak_list)

# Prepare combined positive and negative FASTA files
neg_fasta_files = [
    '../background_negatives_500bp_hg38/HEK293T_DNase_ENCODE.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_validation.fasta',
    '../background_negatives_500bp_hg38/HEK293T_DNase_ENCODE.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_test.fasta',
    '../background_negatives_500bp_hg38/HEK293T_DNase_ENCODE.expand_500bp_hg38_biasaway_GCmatch_background_noOverlap_train.fasta'
]
pos_fasta_files = [
    '../expand_peaks_500bp_fasta_hg38/HEK293T_DNase_ENCODE.narrowPeak.validation.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/HEK293T_DNase_ENCODE.narrowPeak.test.bed.fasta',
    '../expand_peaks_500bp_fasta_hg38/HEK293T_DNase_ENCODE.narrowPeak.train.bed.fasta'
]
output_titles = [
    "../positive_negative_500bp_hg38/HEK293T_DNase_ENCODE.narrowPeak.validation.posneg.fasta",
    "../positive_negative_500bp_hg38/HEK293T_DNase_ENCODE.narrowPeak.test.posneg.fasta",
    "../positive_negative_500bp_hg38/HEK293T_DNase_ENCODE.narrowPeak.train.posneg.fasta"
]

for i, (neg_fasta, pos_fasta, title) in enumerate(zip(neg_fasta_files, pos_fasta_files, output_titles)):
    os.system(f"cat {pos_fasta} {neg_fasta} > {title}")

In [ ]:
import os
import pandas as pd

# Example usage
fasta_file_neg = '../background_negatives_500bp_mm10/Cortex_AgeB_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background.fasta'
fasta_file_pos = '../expand_peaks_500bp_fasta_mm10/Cortex_AgeB_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10.fasta'
output_fasta_file = '../background_negatives_500bp_mm10/Cortex_AgeB_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap.fasta'
peak_file_pos = '../peak_files/Mouse_cortex_ageB/MouseCortexAgeB_all_possible_peaks_merged_expand500bp_eachside.bed'

process_fasta_files(fasta_file_neg, fasta_file_pos, peak_file_pos, output_fasta_file)

# Process and parse the FASTA file
fasta_file = '../background_negatives_500bp_mm10/Cortex_AgeB_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap.fasta'
df = parse_fasta(fasta_file)

# Generate validation set
val = df[df['chr'] == 'chr4']
output_file_validation = '../background_negatives_500bp_mm10/Cortex_AgeB_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_validation.fasta'
df_to_fasta(val, output_file_validation)

# Generate test set
test = df[(df['chr'] == 'chr8') | (df['chr'] == 'chr9')]
output_file_test = '../background_negatives_500bp_mm10/Cortex_AgeB_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_test.fasta'
df_to_fasta(test, output_file_test)

# Generate training set
train = df[~df['chr'].isin(['chr4', 'chr8', 'chr9'])]
output_file_train = '../background_negatives_500bp_mm10/Cortex_AgeB_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_train.fasta'
df_to_fasta(train, output_file_train)

# Prepare narrowPeak files
narrowpeak_list = [
    "../expand_peaks_500bp_mm10/Cortex_AgeB_ATAC_out_ppr.IDR0.1.filt.narrowPeak.train.bed",
    "../expand_peaks_500bp_mm10/Cortex_AgeB_ATAC_out_ppr.IDR0.1.filt.narrowPeak.validation.bed",
    "../expand_peaks_500bp_mm10/Cortex_AgeB_ATAC_out_ppr.IDR0.1.filt.narrowPeak.test.bed"
]
order = [train, val, test]

separate_negatives(narrowpeak_list)

# Prepare combined positive and negative FASTA files
neg_fasta_files = [
    '../background_negatives_500bp_mm10/Cortex_AgeB_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_validation.fasta',
    '../background_negatives_500bp_mm10/Cortex_AgeB_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_test.fasta',
    '../background_negatives_500bp_mm10/Cortex_AgeB_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_train.fasta'
]
pos_fasta_files = [
    '../expand_peaks_500bp_fasta_mm10/Cortex_AgeB_ATAC_out_ppr.IDR0.1.filt.narrowPeak.validation.bed.fasta',
    '../expand_peaks_500bp_fasta_mm10/Cortex_AgeB_ATAC_out_ppr.IDR0.1.filt.narrowPeak.test.bed.fasta',
    '../expand_peaks_500bp_fasta_mm10/Cortex_AgeB_ATAC_out_ppr.IDR0.1.filt.narrowPeak.train.bed.fasta'
]
output_titles = [
    "../positive_negative_500bp_mm10/Cortex_AgeB_ATAC_out_ppr.IDR0.1.filt.narrowPeak.validation.posneg.fasta",
    "../positive_negative_500bp_mm10/Cortex_AgeB_ATAC_out_ppr.IDR0.1.filt.narrowPeak.test.posneg.fasta",
    "../positive_negative_500bp_mm10/Cortex_AgeB_ATAC_out_ppr.IDR0.1.filt.narrowPeak.train.posneg.fasta"
]

for i, (neg_fasta, pos_fasta, title) in enumerate(zip(neg_fasta_files, pos_fasta_files, output_titles)):
    os.system(f"cat {pos_fasta} {neg_fasta} > {title}")

In [ ]:
import os
import pandas as pd

# Example usage
fasta_file_neg = '../background_negatives_500bp_mm10/Cortex_AgeC_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background.fasta'
fasta_file_pos = '../expand_peaks_500bp_fasta_mm10/Cortex_AgeC_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10.fasta'
output_fasta_file = '../background_negatives_500bp_mm10/Cortex_AgeC_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap.fasta'
peak_file_pos = '../peak_files/Mouse_cortex_ageC/MouseCortexAgeC_all_possible_peaks_merged_expand500bp_eachside.bed'

process_fasta_files(fasta_file_neg, fasta_file_pos, peak_file_pos, output_fasta_file)

# Process and parse the FASTA file
fasta_file = '../background_negatives_500bp_mm10/Cortex_AgeC_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap.fasta'
df = parse_fasta(fasta_file)

# Generate validation set
val = df[df['chr'] == 'chr4']
output_file_validation = '../background_negatives_500bp_mm10/Cortex_AgeC_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_validation.fasta'
df_to_fasta(val, output_file_validation)

# Generate test set
test = df[(df['chr'] == 'chr8') | (df['chr'] == 'chr9')]
output_file_test = '../background_negatives_500bp_mm10/Cortex_AgeC_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_test.fasta'
df_to_fasta(test, output_file_test)

# Generate training set
train = df[~df['chr'].isin(['chr4', 'chr8', 'chr9'])]
output_file_train = '../background_negatives_500bp_mm10/Cortex_AgeC_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_train.fasta'
df_to_fasta(train, output_file_train)

# Prepare narrowPeak files
narrowpeak_list = [
    "../expand_peaks_500bp_mm10/Cortex_AgeC_ATAC_out_ppr.IDR0.1.filt.narrowPeak.train.bed",
    "../expand_peaks_500bp_mm10/Cortex_AgeC_ATAC_out_ppr.IDR0.1.filt.narrowPeak.validation.bed",
    "../expand_peaks_500bp_mm10/Cortex_AgeC_ATAC_out_ppr.IDR0.1.filt.narrowPeak.test.bed"
]
order = [train, val, test]

separate_negatives(narrowpeak_list)

# Prepare combined positive and negative FASTA files
neg_fasta_files = [
    '../background_negatives_500bp_mm10/Cortex_AgeC_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_validation.fasta',
    '../background_negatives_500bp_mm10/Cortex_AgeC_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_test.fasta',
    '../background_negatives_500bp_mm10/Cortex_AgeC_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_train.fasta'
]
pos_fasta_files = [
    '../expand_peaks_500bp_fasta_mm10/Cortex_AgeC_ATAC_out_ppr.IDR0.1.filt.narrowPeak.validation.bed.fasta',
    '../expand_peaks_500bp_fasta_mm10/Cortex_AgeC_ATAC_out_ppr.IDR0.1.filt.narrowPeak.test.bed.fasta',
    '../expand_peaks_500bp_fasta_mm10/Cortex_AgeC_ATAC_out_ppr.IDR0.1.filt.narrowPeak.train.bed.fasta'
]
output_titles = [
    "../positive_negative_500bp_mm10/Cortex_AgeC_ATAC_out_ppr.IDR0.1.filt.narrowPeak.validation.posneg.fasta",
    "../positive_negative_500bp_mm10/Cortex_AgeC_ATAC_out_ppr.IDR0.1.filt.narrowPeak.test.posneg.fasta",
    "../positive_negative_500bp_mm10/Cortex_AgeC_ATAC_out_ppr.IDR0.1.filt.narrowPeak.train.posneg.fasta"
]

for i, (neg_fasta, pos_fasta, title) in enumerate(zip(neg_fasta_files, pos_fasta_files, output_titles)):
    os.system(f"cat {pos_fasta} {neg_fasta} > {title}")

In [ ]:
import os
import pandas as pd

# Example usage
fasta_file_neg = '../background_negatives_500bp_mm10/Striatum_AgeC_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background.fasta'
fasta_file_pos = '../expand_peaks_500bp_fasta_mm10/Striatum_AgeC_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10.fasta'
output_fasta_file = '../background_negatives_500bp_mm10/Striatum_AgeC_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap.fasta'
peak_file_pos = '../peak_files/Mouse_striatum_ageC/MouseStriatumAgeC_all_possible_peaks_merged_expand500bp_eachside.bed'

process_fasta_files(fasta_file_neg, fasta_file_pos, peak_file_pos, output_fasta_file)

# Process and parse the FASTA file
fasta_file = '../background_negatives_500bp_mm10/Striatum_AgeC_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap.fasta'
df = parse_fasta(fasta_file)

# Generate validation set
val = df[df['chr'] == 'chr4']
output_file_validation = '../background_negatives_500bp_mm10/Striatum_AgeC_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_validation.fasta'
df_to_fasta(val, output_file_validation)

# Generate test set
test = df[(df['chr'] == 'chr8') | (df['chr'] == 'chr9')]
output_file_test = '../background_negatives_500bp_mm10/Striatum_AgeC_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_test.fasta'
df_to_fasta(test, output_file_test)

# Generate training set
train = df[~df['chr'].isin(['chr4', 'chr8', 'chr9'])]
output_file_train = '../background_negatives_500bp_mm10/Striatum_AgeC_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_train.fasta'
df_to_fasta(train, output_file_train)

# Prepare narrowPeak files
narrowpeak_list = [
    "../expand_peaks_500bp_mm10/Striatum_AgeC_ATAC_out_ppr.IDR0.1.filt.narrowPeak.train.bed",
    "../expand_peaks_500bp_mm10/Striatum_AgeC_ATAC_out_ppr.IDR0.1.filt.narrowPeak.validation.bed",
    "../expand_peaks_500bp_mm10/Striatum_AgeC_ATAC_out_ppr.IDR0.1.filt.narrowPeak.test.bed"
]
order = [train, val, test]

separate_negatives(narrowpeak_list)

# Prepare combined positive and negative FASTA files
neg_fasta_files = [
    '../background_negatives_500bp_mm10/Striatum_AgeC_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_validation.fasta',
    '../background_negatives_500bp_mm10/Striatum_AgeC_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_test.fasta',
    '../background_negatives_500bp_mm10/Striatum_AgeC_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_train.fasta'
]
pos_fasta_files = [
    '../expand_peaks_500bp_fasta_mm10/Striatum_AgeC_ATAC_out_ppr.IDR0.1.filt.narrowPeak.validation.bed.fasta',
    '../expand_peaks_500bp_fasta_mm10/Striatum_AgeC_ATAC_out_ppr.IDR0.1.filt.narrowPeak.test.bed.fasta',
    '../expand_peaks_500bp_fasta_mm10/Striatum_AgeC_ATAC_out_ppr.IDR0.1.filt.narrowPeak.train.bed.fasta'
]
output_titles = [
    "../positive_negative_500bp_mm10/Striatum_AgeC_ATAC_out_ppr.IDR0.1.filt.narrowPeak.validation.posneg.fasta",
    "../positive_negative_500bp_mm10/Striatum_AgeC_ATAC_out_ppr.IDR0.1.filt.narrowPeak.test.posneg.fasta",
    "../positive_negative_500bp_mm10/Striatum_AgeC_ATAC_out_ppr.IDR0.1.filt.narrowPeak.train.posneg.fasta"
]

for i, (neg_fasta, pos_fasta, title) in enumerate(zip(neg_fasta_files, pos_fasta_files, output_titles)):
    os.system(f"cat {pos_fasta} {neg_fasta} > {title}")

In [ ]:
import os
import pandas as pd

# Example usage
fasta_file_neg = '../background_negatives_500bp_mm10/Striatum_AgeB_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background.fasta'
fasta_file_pos = '../expand_peaks_500bp_fasta_mm10/Striatum_AgeB_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10.fasta'
output_fasta_file = '../background_negatives_500bp_mm10/Striatum_AgeB_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap.fasta'
peak_file_pos = '../peak_files/Mouse_striatum_ageB/MouseStriatumAgeB_all_possible_peaks_merged_expand500bp_eachside.bed'

process_fasta_files(fasta_file_neg, fasta_file_pos, peak_file_pos, output_fasta_file)

# Process and parse the FASTA file
fasta_file = '../background_negatives_500bp_mm10/Striatum_AgeB_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap.fasta'
df = parse_fasta(fasta_file)

# Generate validation set
val = df[df['chr'] == 'chr4']
output_file_validation = '../background_negatives_500bp_mm10/Striatum_AgeB_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_validation.fasta'
df_to_fasta(val, output_file_validation)

# Generate test set
test = df[(df['chr'] == 'chr8') | (df['chr'] == 'chr9')]
output_file_test = '../background_negatives_500bp_mm10/Striatum_AgeB_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_test.fasta'
df_to_fasta(test, output_file_test)

# Generate training set
train = df[~df['chr'].isin(['chr4', 'chr8', 'chr9'])]
output_file_train = '../background_negatives_500bp_mm10/Striatum_AgeB_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_train.fasta'
df_to_fasta(train, output_file_train)

# Prepare narrowPeak files
narrowpeak_list = [
    "../expand_peaks_500bp_mm10/Striatum_AgeB_ATAC_out_ppr.IDR0.1.filt.narrowPeak.train.bed",
    "../expand_peaks_500bp_mm10/Striatum_AgeB_ATAC_out_ppr.IDR0.1.filt.narrowPeak.validation.bed",
    "../expand_peaks_500bp_mm10/Striatum_AgeB_ATAC_out_ppr.IDR0.1.filt.narrowPeak.test.bed"
]
order = [train, val, test]

separate_negatives(narrowpeak_list)

# Prepare combined positive and negative FASTA files
neg_fasta_files = [
    '../background_negatives_500bp_mm10/Striatum_AgeB_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_validation.fasta',
    '../background_negatives_500bp_mm10/Striatum_AgeB_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_test.fasta',
    '../background_negatives_500bp_mm10/Striatum_AgeB_ATAC_out_ppr.IDR0.1.filt.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_train.fasta'
]
pos_fasta_files = [
    '../expand_peaks_500bp_fasta_mm10/Striatum_AgeB_ATAC_out_ppr.IDR0.1.filt.narrowPeak.validation.bed.fasta',
    '../expand_peaks_500bp_fasta_mm10/Striatum_AgeB_ATAC_out_ppr.IDR0.1.filt.narrowPeak.test.bed.fasta',
    '../expand_peaks_500bp_fasta_mm10/Striatum_AgeB_ATAC_out_ppr.IDR0.1.filt.narrowPeak.train.bed.fasta'
]
output_titles = [
    "../positive_negative_500bp_mm10/Striatum_AgeB_ATAC_out_ppr.IDR0.1.filt.narrowPeak.validation.posneg.fasta",
    "../positive_negative_500bp_mm10/Striatum_AgeB_ATAC_out_ppr.IDR0.1.filt.narrowPeak.test.posneg.fasta",
    "../positive_negative_500bp_mm10/Striatum_AgeB_ATAC_out_ppr.IDR0.1.filt.narrowPeak.train.posneg.fasta"
]

for i, (neg_fasta, pos_fasta, title) in enumerate(zip(neg_fasta_files, pos_fasta_files, output_titles)):
    os.system(f"cat {pos_fasta} {neg_fasta} > {title}")

In [ ]:
# Example usage
fasta_file_neg = '../background_negatives_500bp_mm10/Pfenning_bulk_CpuCombined_Striatum.expand_500bp_mm10_biasaway_GCmatch_background.fasta'
fasta_file_pos = '../expand_peaks_500bp_fasta_mm10/Pfenning_bulk_CpuCombined_Striatum.expand_500bp_mm10.fasta'
output_fasta_file = '../background_negatives_500bp_mm10/Pfenning_bulk_CpuCombined_Striatum.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap.fasta'

process_fasta_files(fasta_file_neg, fasta_file_pos, output_fasta_file)

# Replace 'fasta_file.fasta' with the path to your actual FASTA file
fasta_file = '../background_negatives_500bp_mm10/Pfenning_bulk_CpuCombined_Striatum.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap.fasta'  # Example: '/path/to/your/fasta_file.fasta'
df = parse_fasta(fasta_file)

val = df[df['chr'] == 'chr4']
output_file = '../background_negatives_500bp_mm10/Pfenning_bulk_CpuCombined_Striatum.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_validation.fasta'   # Specify the output file name
df_to_fasta(val, output_file)

test = df[(df['chr'] == 'chr8')|(df['chr']=='chr9')]
# Example usage
output_file = '../background_negatives_500bp_mm10/Pfenning_bulk_CpuCombined_Striatum.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_test.fasta'   # Specify the output file name
df_to_fasta(test, output_file)

train = df[~df['chr'].isin(['chr4','chr8','chr9'])]
output_file = '../background_negatives_500bp_mm10/Pfenning_bulk_CpuCombined_Striatum.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_train.fasta'   # Specify the output file name
df_to_fasta(train, output_file)

narrowpeak_list = ["../expand_peaks_500bp_mm10/Pfenning_bulk_CpuCombined_Striatum.narrowPeak.train.bed",
"../expand_peaks_500bp_mm10/Pfenning_bulk_CpuCombined_Striatum.narrowPeak.validation.bed",
"../expand_peaks_500bp_mm10/Pfenning_bulk_CpuCombined_Striatum.narrowPeak.test.bed"]
order = [train,val,test]
separate_negatives(narrowpeak_list)
for i in range(len(narrowpeak_list)):
    df_tmp = pd.read_csv(narrowpeak_list[i],sep='\t',header=None)
    current_neg = order[i]
    current_neg = current_neg[['chr','start','end']]
    current_neg[6]=0
    current_neg.columns = [0,1,2,6]
    current_neg[[3,4,5,7,8,9]] = "NA"
    current_neg = current_neg[[0,1,2,3,4,5,6,7,8,9]]
    pd.concat([df_tmp,current_neg]).to_csv(narrowpeak_list[i][:-4]+".withNeg.bed",sep='\t',header=None,index=None)

import os
neg_fasta = ['../background_negatives_500bp_mm10/Pfenning_bulk_CpuCombined_Striatum.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_validation.fasta',   # Specify the output file name
'../background_negatives_500bp_mm10/Pfenning_bulk_CpuCombined_Striatum.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_test.fasta',   # Specify the output file name
'../background_negatives_500bp_mm10/Pfenning_bulk_CpuCombined_Striatum.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_train.fasta']   # Specify the output file name
pos_fasta = ['../expand_peaks_500bp_fasta_mm10/Pfenning_bulk_CpuCombined_Striatum.narrowPeak.validation.bed.fasta',
             '../expand_peaks_500bp_fasta_mm10/Pfenning_bulk_CpuCombined_Striatum.narrowPeak.test.bed.fasta',
             '../expand_peaks_500bp_fasta_mm10/Pfenning_bulk_CpuCombined_Striatum.narrowPeak.train.bed.fasta'
             ]
title = ["../positive_negative_500bp_mm10/Pfenning_bulk_CpuCombined_Striatum.narrowPeak.validation.posneg.fasta",
         "../positive_negative_500bp_mm10/Pfenning_bulk_CpuCombined_Striatum.narrowPeak.test.posneg.fasta",
         "../positive_negative_500bp_mm10/Pfenning_bulk_CpuCombined_Striatum.narrowPeak.train.posneg.fasta"]
         
for i in range(len(neg_fasta)):
    current_neg_fasta = neg_fasta[i]
    current_pos_fasta = pos_fasta[i]
    os.system(f"cat {current_pos_fasta} {current_neg_fasta} > {title[i]}")


In [ ]:
# Example usage
fasta_file_neg = '../background_negatives_500bp_mm10/Pfenning_bulk_CtxCombined_Cortex.expand_500bp_mm10_biasaway_GCmatch_background.fasta'
fasta_file_pos = '../expand_peaks_500bp_fasta_mm10/Pfenning_bulk_CtxCombined_Cortex.expand_500bp_mm10.fasta'
output_fasta_file = '../background_negatives_500bp_mm10/Pfenning_bulk_CtxCombined_Cortex.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap.fasta'

process_fasta_files(fasta_file_neg, fasta_file_pos, output_fasta_file)

# Replace 'fasta_file.fasta' with the path to your actual FASTA file
fasta_file = '../background_negatives_500bp_mm10/Pfenning_bulk_CtxCombined_Cortex.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap.fasta'  # Example: '/path/to/your/fasta_file.fasta'
df = parse_fasta(fasta_file)

val = df[df['chr'] == 'chr4']
output_file = '../background_negatives_500bp_mm10/Pfenning_bulk_CtxCombined_Cortex.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_validation.fasta'   # Specify the output file name
df_to_fasta(val, output_file)

test = df[(df['chr'] == 'chr8')|(df['chr']=='chr9')]
# Example usage
output_file = '../background_negatives_500bp_mm10/Pfenning_bulk_CtxCombined_Cortex.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_test.fasta'   # Specify the output file name
df_to_fasta(test, output_file)

train = df[~df['chr'].isin(['chr4','chr8','chr9'])]
output_file = '../background_negatives_500bp_mm10/Pfenning_bulk_CtxCombined_Cortex.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_train.fasta'   # Specify the output file name
df_to_fasta(train, output_file)

narrowpeak_list = ["../expand_peaks_500bp_mm10/Pfenning_bulk_CtxCombined_Cortex.narrowPeak.train.bed",
"../expand_peaks_500bp_mm10/Pfenning_bulk_CtxCombined_Cortex.narrowPeak.validation.bed",
"../expand_peaks_500bp_mm10/Pfenning_bulk_CtxCombined_Cortex.narrowPeak.test.bed"]
order = [train,val,test]
for i in range(len(narrowpeak_list)):
    df_tmp = pd.read_csv(narrowpeak_list[i],sep='\t',header=None)
    current_neg = order[i]
    current_neg = current_neg[['chr','start','end']]
    current_neg[6]=0
    current_neg.columns = [0,1,2,6]
    current_neg[[3,4,5,7,8,9]] = "NA"
    current_neg = current_neg[[0,1,2,3,4,5,6,7,8,9]]
    pd.concat([df_tmp,current_neg]).to_csv(narrowpeak_list[i][:-4]+".withNeg.bed",sep='\t',header=None,index=None)

import os
neg_fasta = ['../background_negatives_500bp_mm10/Pfenning_bulk_CtxCombined_Cortex.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_validation.fasta',   # Specify the output file name
'../background_negatives_500bp_mm10/Pfenning_bulk_CtxCombined_Cortex.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_test.fasta',   # Specify the output file name
'../background_negatives_500bp_mm10/Pfenning_bulk_CtxCombined_Cortex.expand_500bp_mm10_biasaway_GCmatch_background_noOverlap_train.fasta']   # Specify the output file name
pos_fasta = ['../expand_peaks_500bp_fasta_mm10/Pfenning_bulk_CtxCombined_Cortex.narrowPeak.validation.bed.fasta',
             '../expand_peaks_500bp_fasta_mm10/Pfenning_bulk_CtxCombined_Cortex.narrowPeak.test.bed.fasta',
             '../expand_peaks_500bp_fasta_mm10/Pfenning_bulk_CtxCombined_Cortex.narrowPeak.train.bed.fasta'
             ]
title = ["../positive_negative_500bp_mm10/Pfenning_bulk_CtxCombined_Cortex.narrowPeak.validation.posneg.fasta",
         "../positive_negative_500bp_mm10/Pfenning_bulk_CtxCombined_Cortex.narrowPeak.test.posneg.fasta",
         "../positive_negative_500bp_mm10/Pfenning_bulk_CtxCombined_Cortex.narrowPeak.train.posneg.fasta"]
         
for i in range(len(neg_fasta)):
    current_neg_fasta = neg_fasta[i]
    current_pos_fasta = pos_fasta[i]
    os.system(f"cat {current_pos_fasta} {current_neg_fasta} > {title[i]}")
